# Convolution Theorem: Two Square Pulses


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-15.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

Convolving two rects of width $d$ gives a triangle of width $2d$. Multiplying their
transforms, two sincs, gives a sinc squared. **These are the same statement**, and the
grid below shows both routes side by side.

`ch05-14` slides the pulses across each other to build the triangle directly. This
notebook takes the other road: transform, multiply, and note that you never have to
slide anything at all. That is what makes the convolution theorem worth having.

## Import libraries

`ConnectionPatch` is the only unusual import. It draws the arrows *between* subplots that
turn six separate axes into a single flow diagram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch


In [ ]:
%matplotlib inline
# ==========================================
# 1. Configuration & Styling
# ==========================================
plot_params = {
    'font.size': 14,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Define parameters

**The two shapes** are the same rect of width $d$, so the answer comes out as clean as
possible: a triangle in space and a sinc squared in frequency.

**The frequency axis.** `k` is the *angular* spatial frequency $k = 2\pi/\lambda$, which is
the convention this text uses, so no $2\pi$ ever appears in an exponent. That is why the
axis is built as `2 * np.pi * np.fft.fftfreq(...)` rather than `fftfreq` alone.

**The transforms are analytic, not numerical.** From the transform table,
$\mathrm{rect}(x/d) \rightarrow d\,\mathrm{sinc}(kd/2)$, so the bottom row is drawn from
that formula and comes out perfectly smooth.

Watch the sinc convention. This text uses the **unnormalized** $\mathrm{sinc}(u) =
\sin(u)/u$, while `np.sinc` is the **normalized** $\sin(\pi z)/(\pi z)$. Dividing the
argument by $\pi$ converts between them. Get this wrong and the sinc comes out $\pi$
times too wide, which looks entirely plausible on a plot.

**The last line is the whole point.** `G_val = F_val * H_val` multiplies. It never
convolves anything.

In [ ]:
# ==========================================
# 2. Data Generation
# ==========================================
# Both pulses are the same rect of width d, so the answer is the cleanest
# possible pair: a triangle in space, a sinc squared in frequency.
width1 = 2.0                       # the pulse width d

N = 8192
L = 40.0
x = np.linspace(-L / 2, L / 2, N, endpoint=False)
dx = x[1] - x[0]

# Angular spatial frequency, matching this text's convention F(k) = int f e^{-ikx} dx
k = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(N, d=dx))


def get_rect(t):
    """A rect of width d, centered on the origin."""
    return np.where(np.abs(t) <= width1 / 2, 1.0, 0.0)


f_val = get_rect(x)
h_val = get_rect(x)

# --- Space domain: the convolution ---
g_val = np.convolve(f_val, h_val, mode='same') * dx

# --- Frequency domain: the analytic transforms ---
# From the Table: rect(x/d)  ->  d sinc(kd/2), with sinc(u) = sin(u)/u.
# NumPy's np.sinc is the NORMALIZED sinc, sin(pi z)/(pi z), so the argument
# needs dividing by pi to match this text's unnormalized convention.
F_val = width1 * np.sinc(k * width1 / (2 * np.pi))
H_val = width1 * np.sinc(k * width1 / (2 * np.pi))

# The convolution theorem: multiply, do not convolve
G_val = F_val * H_val


## Plot

A 2x3 grid, read as a flow diagram.

The **top row is the space domain**: $f$, then $h$, then their convolution $g$. The
$\otimes$ between the first two columns marks the operation, and the annotated $2d$ on the
triangle shows that convolving two functions of width $d$ gives a result of width $2d$.

The **bottom row is the frequency domain**: the same three functions, transformed. Here
the operation between the first two columns is $\times$, plain multiplication.

The **vertical arrows** are Fourier transforms. That is the claim the whole figure makes:
you can go across the top and then down, or down and then across, and arrive at the same
place. Convolving in space and multiplying in frequency are two routes to one answer.

The sinc squared in the bottom right is worth a second look. It never goes negative, and
its central lobe is narrower than either sinc that produced it, because multiplying two
functions that are both less than one everywhere except the peak suppresses the tails.

In [ ]:
# ==========================================
# 3. Plotting the Grid
# ==========================================
fig1, axes1 = plt.subplots(2, 3, figsize=(16, 9))
plt.subplots_adjust(wspace=0.50, hspace=0.78, left=0.13, right=0.97,
                    top=0.90, bottom=0.08)

xlim1 = 3.5
klim1 = 12.0

# --- Top Row: Space Domain ---
ax_f = axes1[0, 0]
ax_f.plot(x, f_val, 'r')
ax_f.set_title(r"$f(x) = \mathrm{rect}(x/d)$", fontsize=16, color='red')
ax_f.set_xlim(-xlim1, xlim1)
ax_f.set_ylim(-0.15, 1.3)
ax_f.set_xlabel(r"$x$")
apply_custom_formatting(ax_f)

ax_h = axes1[0, 1]
ax_h.plot(x, h_val, 'b')
ax_h.set_title(r"$h(x) = \mathrm{rect}(x/d)$", fontsize=16, color='blue')
ax_h.set_xlim(-xlim1, xlim1)
ax_h.set_ylim(-0.15, 1.3)
ax_h.set_xlabel(r"$x$")
apply_custom_formatting(ax_h)

ax_g = axes1[0, 2]
ax_g.plot(x, g_val, 'g')
ax_g.fill_between(x, 0, g_val, color='green', alpha=0.2)
ax_g.set_title(r"$g(x) = f \otimes h = d\,\mathrm{tri}(x/d)$", fontsize=16, color='green')
ax_g.set_xlim(-xlim1, xlim1)
ax_g.set_ylim(-0.15, width1 * 1.3)
ax_g.set_xlabel(r"$x$")
ax_g.annotate("", xy=(-width1, 0.14), xytext=(width1, 0.14),
              arrowprops=dict(arrowstyle="<->", lw=1.5, color='green'))
ax_g.text(0, 0.22, r"$2d$", ha='center', color='green', fontsize=14)
apply_custom_formatting(ax_g)

# --- Bottom Row: Frequency Domain ---
ax_F = axes1[1, 0]
ax_F.plot(k, F_val, 'darkred')
ax_F.set_title(r"$F(k) = d\,\mathrm{sinc}(kd/2)$", fontsize=16, color='darkred')
ax_F.set_xlim(-klim1, klim1)
ax_F.set_xlabel(r"$k$")
apply_custom_formatting(ax_F)

ax_H = axes1[1, 1]
ax_H.plot(k, H_val, 'darkblue')
ax_H.set_title(r"$H(k) = d\,\mathrm{sinc}(kd/2)$", fontsize=16, color='darkblue')
ax_H.set_xlim(-klim1, klim1)
ax_H.set_xlabel(r"$k$")
apply_custom_formatting(ax_H)

ax_G = axes1[1, 2]
ax_G.plot(k, G_val, 'darkgreen')
ax_G.fill_between(k, 0, G_val, color='green', alpha=0.2)
ax_G.set_title(r"$G(k) = F \cdot H = d^2\mathrm{sinc}^2(kd/2)$",
               fontsize=16, color='darkgreen')
ax_G.set_xlim(-klim1, klim1)
ax_G.set_xlabel(r"$k$")
apply_custom_formatting(ax_G)

# ==========================================
# 4. Adding Flowchart Annotations
# ==========================================
# Positions are read back from the axes so the glyphs land in the gutters
# between columns rather than on top of a panel.
fig1.canvas.draw()


def draw_arrow(ax_from, ax_to, text=""):
    """Vertical arrow between a column's two panels, as in ch05-10.

    Axes-fraction coordinates with axesA/axesB, added through add_artist.
    Passing figure-fraction coords together with axesA is not valid and
    raises AttributeError on transFigure.
    """
    con = ConnectionPatch(xyA=(0.5, -0.32), xyB=(0.5, 1.14),
                          coordsA="axes fraction", coordsB="axes fraction",
                          axesA=ax_from, axesB=ax_to,
                          arrowstyle="-|>", shrinkB=5, linewidth=2.5,
                          color="black")
    ax_from.add_artist(con)
    if text:
        ax_to.text(0.56, 1.20, text, transform=ax_to.transAxes,
                   ha='left', va='bottom', fontsize=15)


draw_arrow(ax_f, ax_F, text="F.T.")
draw_arrow(ax_h, ax_H, text="F.T.")
draw_arrow(ax_g, ax_G, text="F.T.")


def gutter(ax_left, ax_right):
    """Center of the CLEAR gap between two side-by-side panels.

    The right panel's y-tick labels stick out to the left of its axes box, so
    centering between the boxes alone drops the glyph on top of them. Measuring
    where those labels actually begin gives a gap that is genuinely empty, and
    removes the need for a hand-tuned nudge.
    """
    fig = ax_left.figure
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    right_x0 = ax_right.get_position().x0
    for lab in ax_right.get_yticklabels():
        if lab.get_visible() and lab.get_text():
            right_x0 = min(
                right_x0,
                lab.get_window_extent(renderer).transformed(inv).x0)
    return (ax_left.get_position().x1 + right_x0) / 2


def row_mid(ax):
    p = ax.get_position()
    return (p.y0 + p.y1) / 2


# Operator glyphs, centered in the gutters
gx1, gx2 = gutter(ax_f, ax_h), gutter(ax_h, ax_g)
y_top, y_bot = row_mid(ax_f), row_mid(ax_F)

# One x for both the operator word and its symbol, so they stay aligned.
# gutter() already accounts for the tick labels, so no nudge is wanted here:
# the old one pushed the word onto the left panel's frame.
op_x = gx1


fig1.text(op_x, y_top + 0.055, "convolve", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_top, r"$\otimes$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_top, r"$=$", fontsize=32, ha='center', va='center')

fig1.text(op_x, y_bot + 0.055, "multiply", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_bot, r"$\times$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_bot, r"$=$", fontsize=32, ha='center', va='center')

# Row labels, to the left of the first column
fig1.text(0.045, y_top, "Space\nDomain", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')
fig1.text(0.045, y_bot, "Frequency\nDomain", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.

In [ ]:
# fig1.savefig("../figures/ch05/ch05-15-convolution-theorem.jpg", dpi=200, bbox_inches='tight')
